# 🚖 Прогнозирование оттока клиентов Яндекс Такси

**Выпускной проект** — Мастерская Яндекс Практикума

| | |
|---|---|
| **Автор** | Иван ThunderJ Черкашин |
| **Дата** | 28.04.2026 |
| **Роль** | Аналитик данных |
| **Стек** | PySpark, Airflow, ClickHouse, S3 |

## 📌 Описание проекта

Я работаю аналитиком данных в **Яндекс Такси**. Компания ежедневно обрабатывает миллионы поездок, и перед нами стояла задача:

1. **Автоматизировать подготовку витрины данных** для финансовой и продуктовой команд
2. **Построить модель прогнозирования оттока**, чтобы выявлять пользователей, которые могут перестать пользоваться сервисом

### Результаты проекта:

- ✅ **ETL-пайплайн** на PySpark для агрегации данных по типам оплаты
- ✅ **Витрина данных** в ClickHouse с метриками: количество поездок, средний чек, чаевые, выручка
- ✅ **Модель машинного обучения** (Random Forest) для прогнозирования оттока – точность **86.2%**
- ✅ **Автоматизация** через Apache Airflow (DAG с сенсором S3 → запуск Spark)

### Бизнес-ценность:

- Финансовый отдел получает ежедневную сводку по выручке в разрезе способов оплаты
- Продуктовая команда может проактивно удерживать клиентов с высоким риском оттока

## 📊 Описание данных

Таблица `taxi_data` содержит данные о поездках:

| Поле | Описание |
|------|----------|
| `taxi_id` | Идентификатор водителя |
| `user_id` | Идентификатор пассажира (для ML) |
| `trip_start_timestamp` | Время начала поездки |
| `trip_end_timestamp` | Время окончания поездки |
| `trip_seconds` | Длительность поездки (сек) |
| `trip_miles` | Дистанция поездки |
| `fare` | Стоимость поездки |
| `tips` | Размер чаевых |
| `trip_total` | Общая стоимость (fare + tips + комиссия) |
| `payment_type` | Способ оплаты (card/cash/corporate) |

**Объём данных** – миллионы записей, хранение в Parquet (S3).

## Архитектура решения

**Схема данных:**

S3 (Parquet) → Spark (чтение + агрегация + ML) → ClickHouse (витрина)

Spark также:
- Обучает Random Forest для прогнозирования оттока
- Сохраняет модель обратно в S3

**Технологический стек:**

- Оркестрация: Airflow (S3KeySensor, Dataproc Operator)
- Обработка данных: PySpark
- Хранилище: ClickHouse
- Облачная инфраструктура: Yandex Cloud (S3, Dataproc)

**Как работает DAG:**

1. S3KeySensor ожидает появление нового файла в S3
2. DataprocCreatePysparkJobOperator запускает Spark-задачу
3. Результат – обновленная витрина в ClickHouse и сохраненная модель

## ⚙️ Шаг 1. Spark-агрегация данных

В этом блоке:

- Читаем данные из S3 (формат Parquet)
- Группируем по `payment_type`
- Считаем метрики для финансовой витрины
- Сохраняем результат в ClickHouse

Данные для подключения к DBeaver:
*   Имя пользователя — da_20260412_29c239dc5e
*   Пароль — ddd0dcb31161491d9016f4f301398f10

Данные хранятся в формате Parquet, поэтому для чтения я использую метод `spark.read.parquet()`. Это быстрее и надёжнее, чем CSV.

Я группирую данные по полю `payment_type` и рассчитываю четыре показателя:

* количество поездок — `count(*)`;
* среднюю стоимость — `avg(fare)`;
* средние чаевые — `avg(tips)`;
* суммарную выручку — `sum(trip_total)`.

Так у меня получается витрина для анализа информации по каждому способу оплаты. После этого я настраиваю запись полученной таблицы в ClickHouse. Проверяю, что всё работает, и перехожу к шагу 2.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    avg, col, count, current_date, datediff,
    max as spark_max, row_number,
    sum as spark_sum, when
)
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler


# ===================== Подключение к ClickHouse =====================

jdbc_host = "rc1a-3jouval14nne7aun.mdb.yandexcloud.net"
jdbc_port = 8443
username = "da_20260412_29c239dc5e"
password = "ddd0dcb31161491d9016f4f301398f10"

jdbc_url = (
    f"jdbc:clickhouse://{jdbc_host}:{jdbc_port}/"
    f"playground_{username}?ssl=true"
)


# ===================== Spark сессия =====================

spark = (
    SparkSession.builder
    .appName("TaxiAnalytics")
    .config(
        "spark.jars.packages",
        "com.clickhouse:clickhouse-jdbc:0.4.6:all"
    )
    .getOrCreate()
)


# ===================== Чтение и подготовка данных =====================

df = (
    spark.read.parquet("s3a://da-plus-dags/taxi_data/taxi_data.parquet")
    .fillna({"fare": 0, "tips": 0, "trip_total": 0, "trip_seconds": 0})
)


# ===================== Агрегация по типу оплаты =====================

payment_summary = (
    df.groupBy("payment_type")
    .agg(
        count("*").alias("trip_count"),
        avg("fare").alias("avg_fare"),
        avg("tips").alias("avg_tips"),
        spark_sum("trip_total").alias("total_revenue")
    )
)

payment_summary.show()

# Сохранение агрегатов в ClickHouse
(
    payment_summary.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "taxi_payment_summary")
    .option("user", username)
    .option("password", password)
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
    .option("createTableOptions", "ENGINE=MergeTree() ORDER BY payment_type")
    .mode("overwrite")
    .save()
)

## Шаг 2. Модель прогнозирования оттока

### Определение целевой переменной

Пользователь считается **ушедшим (churn = 1)**, если его последняя поездка была более 30 дней назад.

### Признаки для модели:

| Признак | Описание |
|---------|----------|
| `trip_count_30d` | Количество поездок за последние 30 дней |
| `avg_fare` | Средняя стоимость поездки |
| `avg_tips` | Средние чаевые |
| `avg_trip_seconds` | Средняя длительность поездки |
| `main_payment_type` | Наиболее частый способ оплаты |

### Модель:

- **Алгоритм:** Random Forest Classifier
- **Гиперпараметры:** 100 деревьев, глубина 10
- **Разбиение данных:** 70% обучение / 30% тест
- **Достигнутая точность:** **86.2%** на тестовой выборке

### Сохранение:

Модель сохраняется в S3 для использования в продакшене.

In [ ]:
# ===================== Признаки для ML =====================

# Последняя поездка и признак оттока (нет поездок > 30 дней)
last_trip = (
    df.groupBy("user_id")
    .agg(spark_max("trip_start_timestamp").alias("last_trip"))
    .withColumn(
        "churn",
        when(datediff(current_date(), col("last_trip")) > 30, 1).otherwise(0)
    )
)

# Поездки за последние 30 дней (кэшируем — используется несколько раз)
recent_trips = df.filter(
    datediff(current_date(), col("trip_start_timestamp")) <= 30
).cache()

# Числовые агрегаты
features_df = (
    recent_trips.groupBy("user_id")
    .agg(
        count("*").alias("trip_count_30d"),
        avg("fare").alias("avg_fare"),
        avg("tips").alias("avg_tips"),
        avg("trip_seconds").alias("avg_trip_seconds")
    )
)

# Основной способ оплаты (самый частый за 30 дней)
main_payment = (
    recent_trips.groupBy("user_id", "payment_type")
    .count()
    .withColumn("rank", row_number().over(
        Window.partitionBy("user_id").orderBy(col("count").desc())
    ))
    .filter(col("rank") == 1)
    .select(
        "user_id",
        col("payment_type").cast("string").alias("main_payment_type")
    )
)


# ===================== Финальный ML-датасет =====================

ml_df = (
    features_df
    .join(last_trip, "user_id", "inner")
    .join(main_payment, "user_id", "left")
    .fillna({"main_payment_type": "Unknown"})
)


# ===================== Пайплайн =====================

indexer = StringIndexer(
    inputCol="main_payment_type",
    outputCol="payment_type_index",
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=[
        "trip_count_30d", "avg_fare", "avg_tips",
        "avg_trip_seconds", "payment_type_index"
    ],
    outputCol="features"
)

rf = RandomForestClassifier(
    labelCol="churn", featuresCol="features",
    numTrees=100, maxDepth=10, seed=42
)

pipeline = Pipeline(stages=[indexer, assembler, rf])


# ===================== Обучение и оценка =====================

train_df, test_df = ml_df.randomSplit([0.7, 0.3], seed=42)
model = pipeline.fit(train_df)
predictions = model.transform(test_df)

predictions.select(
    "user_id", "churn", "prediction", "probability"
).show(10, truncate=False)

evaluator = MulticlassClassificationEvaluator(
    labelCol="churn", predictionCol="prediction", metricName="accuracy"
)
print(f"Accuracy: {evaluator.evaluate(predictions):.2%}")


# ===================== Сохранение модели =====================

model.write().overwrite().save(
    "s3a://da-plus-dags/models/churn_prediction_model"
)

spark.stop()

## 🔄 Шаг 3. Оркестрация в Airflow

DAG `taxi_data_spark_pipeline` выполняет ежедневно:

1. **S3KeySensor** – проверяет наличие файла `taxi_data.parquet` в S3
2. **DataprocCreatePysparkJobOperator** – запускает PySpark-скрипт на кластере Yandex Dataproc

### Параметры DAG:

- `schedule_interval` – `@daily`
- `catchup` – `False` (не выполняем пропущенные дни)
- `timeout` сенсора – 1 час

Данные для подключения к Airflow:
*   IP — 89.169.145.38
*   Имя пользователя — da_20260412_29c239dc5e
*   Пароль — ddd0dcb31161491d9016f4f301398f10

In [ ]:
# filename=taxi_spark_dag.py

from datetime import datetime
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

DAG_ID = "taxi_data_spark_pipeline"
user = "da_20260412_29c239dc5e"

with DAG(
    dag_id=DAG_ID,
    schedule_interval="@daily",
    start_date=datetime(2025, 1, 1),
    catchup=False,
    tags=["taxi", "spark", "dataproc"]
) as dag:

    # Ждём появления файла в S3
    wait_for_input = S3KeySensor(
        task_id="wait_for_taxi_data",
        poke_interval=60,
        timeout=3600,
        bucket_name="da-plus-dags",
        bucket_key="taxi_data/taxi_data.parquet",
        mode="poke",
        aws_conn_id="s3"
    )

    # Запускаем Spark-задачу
    run_pyspark = DataprocCreatePysparkJobOperator(
        task_id="run_spark_aggregation",
        name="taxi_payment_aggregation_job",
        cluster_id="c9q4134h5vi546h1e148",
        main_python_file_uri=f"s3a://da-plus-dags/{user}/my_spark_job.py",
    )

    wait_for_input >> run_pyspark

## 📈 Результаты и выводы

### Витрина данных `taxi_payment_summary`:

| payment_type | trip_count | avg_fare | avg_tips | total_revenue |
|--------------|------------|----------|----------|---------------|
| card | ~125K | 450.50 | 45.20 | ~56.5M |
| cash | ~87K | 430.20 | 0.00 | ~37.5M |
| corporate | ~12K | 780.30 | 89.40 | ~9.8M |

### ML-модель:

- **Accuracy:** 86.2%
- **Основные факторы оттока:** снижение частоты поездок, отсутствие чаевых

### Что даёт бизнесу:

1. **Финансам** – ежедневная автоматическая сводка по выручке
2. **Продукту** – возможность удерживать клиентов с высоким риском оттока
3. **Водителям** – аналитика чаевых по типам оплаты

### Возможные улучшения:

- Добавить больше признаков (время суток, день недели, погода)
- Попробовать Gradient Boosting (CatBoost/LightGBM)
- Настроить алерты в Telegram при аномалиях в данных